# Bitácoras Onmásticas del SEMEFO-DF de 1968-1982
# Hallazgos 1: 

In [1]:
import os
import re
import json
from glob import glob

import pandas as pd
from numpy import nan
from unidecode import unidecode 

def simplify_string(text, text_format='upper'):
    if type(text)==str:
        if text_format=='upper': text = text.upper()
        if text_format=='lower': text = text.lower()
        if text_format=='title': text = text.title()
        if text_format=='capitalize': text = text.capitalize()
        text = unidecode(text)
        text = ' '.join([t for t in text.split()])
    return text

## Datos de bitácora onomástica

In [2]:
file_data = "data_clean/SEMEFO_BitacoraOnomastica_1974_review.csv"
df = pd.read_csv(file_data, dtype='str')

bo_nombres = df.loc[df['TIPOS DE RESTOS']=='Cadáver', 
                    ['NOMBRES PROPIOS', 'APELLIDO PATERNO', 'APELLIDO MATERNO']  ]
bo_nombres = bo_nombres.dropna(how='any')
bo_nombres = bo_nombres.apply(lambda s: ' '.join(s), axis=1)
bo_nombres = bo_nombres.apply(simplify_string)
bo_nombres = bo_nombres.sort_values().drop_duplicates().to_list()
bo_nombres[0:10]

['A ALVARO FLORES AGUILAR',
 'A BENIGNO SOLIS GOMEZ',
 'A CAJETANA HERNANDEZ RUPERTO',
 'A EDIVIER GAMIDO RAMIREZ',
 'A ENRIQUE MENESES VARGAS',
 'A IVONNE PEREZ MENDEZ',
 'A JORGE E GOMEZ DE LA PLATA',
 'A LEONARDO AVILA TEJADILLA',
 'A PATRICIA CADENA BELLO',
 'ABDIAS AVILES JUAN']

In [3]:
## Datos de bitácora angelus

In [4]:
file_json = "data_raw/angelus_vicdirecta_20230306.json"
api_onto = "https://ontologias.pruebascnb.org/v1.0/"

with open(file_json) as f:
    data = json.load(f)
    data = data['entities_with_labels']
    
ang_nombres = []

for entity in data:
    label = entity['labels']
    nombres, paterno, materno = None, None, None
    for anot in label:
        if 'tieneNombresPropios' in anot['label_predicate']:
            nombres = anot['label_value']
        if 'tieneApellidoPaterno' in anot['label_predicate']:
            paterno = anot['label_value']
        if 'tieneApellidoMaterno' in anot['label_predicate']:
            materno = anot['label_value']
        try: 
            nombre = nombres +' ' +paterno +' ' + materno
            nombre = simplify_string(nombre)
            ang_nombres.append(nombre)
        except TypeError: pass

ang_nombres = list(set(ang_nombres))
ang_nombres.sort()
ang_nombres[0:10], len(ang_nombres)

(['AARON DE MEZA PADILLA',
  'ABAD TORRES MEZA',
  'ABDAYAN GUZMAN CRUZ',
  'ABEL ALMAZAN SALDANA',
  'ABEL MORALES MARTINEZ',
  'ABEL SALGADO VALDEZ',
  'ABEL SANCHEZ PATINO',
  'ABELARDO MORALES GERVACIO',
  'ABRAHAN JIMENEZ GARCIA',
  'ABUNDIO ONOFRE CAMPO'],
 1245)

## Comparar

In [5]:
from thefuzz import fuzz

select_thr = 95

matrix_compare = list()
high_similarity = list()

for nombre_bitacora in bo_nombres:
    row = []
    for nombre_angelus in ang_nombres:
        ratio = fuzz.token_sort_ratio(nombre_bitacora, nombre_angelus)
        row.append( ratio )
        if ratio>=select_thr:
            high_similarity.append( (nombre_bitacora, nombre_angelus, ratio) )
    matrix_compare.append(row)

matrix_compare = pd.DataFrame(matrix_compare, columns=ang_nombres, index=bo_nombres)
high_similarity = pd.DataFrame(high_similarity, columns=['nombre_bitacora','nombre_angelus', 'ratio'])
high_similarity = high_similarity.sort_values(by='ratio', ascending=False)

high_similarity

,nombre_bitacora,nombre_angelus,ratio
7,JOSE LUIS MARTINEZ GARCIA,JOSE LUIS MARTINEZ GARCIA,100
3,J AURELIO MORALES GONZALEZ,AURELIO GONZALEZ MORALES,96
8,JUAN MARTINEZ HERNANDEZ,JULIAN HERNANDEZ MARTINEZ,96
4,JOSE FLORES GARCIA,JOSUEL FLORES GARCIA,95
2,FRANCISCO JAVIER OLMERA MARTINEZ,FRANCISCO JAVIER MARTINEZ MEJIA,92
5,JOSE GONZALEZ LOPEZ,JORGE GONZALEZ LOPEZ,92
0,ALBERTO BARRERA LOPEZ,ALBERTO LOPEZ HERRERA,90
1,DANIEL MARTINEZ LARA,DANIEL MARTINEZ GARCIA,90
6,JOSE GONZALEZ RODRIGUEZ,JUAN JOSE RODRIGUEZ GONZALEZ,90
9,SALVADOR ALVAREZ MARTINEZ,SALVADOR ALFARO MARTINEZ,90


In [6]:
matrix_file = "data_clean/BO_1974_matchmatrix_20230306.csv"
highsim_file = "data_clean/BO_1974_highsimilarity_20230306.csv"

matrix_compare.to_csv(matrix_file)
high_similarity.to_csv(highsim_file, index=False)

**Done!**